In [16]:
import time
import tracemalloc
import cProfile
from cs336_basics.bpe import train_bpe
from tests.common import gpt2_bytes_to_unicode


In [17]:
# input_path = "../data/TinyStoriesV2-GPT4-valid.txt"
input_path = "../data/TinyStoriesV2-GPT4-train.txt"
vocab_size = 10000
special_tokens = ["<|endoftext|>"]

In [18]:
tracemalloc.start()

start = time.perf_counter()

vocab, merges = train_bpe(
    input_path,
    vocab_size,
    special_tokens
)

duration = time.perf_counter() - start
print("Duration: ", duration)

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')

print("[ Top 10 ]")
for stat in top_stats[:10]:
    print(stat)


Duration:  246.93309979999322
[ Top 10 ]
C:\Users\74729\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\multiprocessing\connection.py:256: size=1092 KiB, count=9905, average=113 B
G:\CS336\assignment1-basics\cs336_basics\bpe.py:225: size=835 KiB, count=11244, average=76 B
G:\CS336\assignment1-basics\cs336_basics\bpe.py:189: size=554 KiB, count=9745, average=58 B
g:\CS336\assignment1-basics\.venv\Lib\site-packages\IPython\core\compilerop.py:178: size=453 KiB, count=4336, average=107 B
G:\CS336\assignment1-basics\cs336_basics\bpe.py:185: size=370 KiB, count=9744, average=39 B
C:\Users\74729\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\tracemalloc.py:558: size=298 KiB, count=5336, average=57 B
G:\CS336\assignment1-basics\cs336_basics\bpe.py:215: size=281 KiB, count=5132, average=56 B
<frozen importlib._bootstrap_external>:781: size=257 KiB, count=2117, average=124 B
G:\CS336\assignment1-basics\cs336_basics\bpe.py:208: size=232 KiB, count=4251, average=56 B

In [19]:
import json
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

d = gpt2_bytes_to_unicode()
vocab_unicode = {}
for key, value in vocab.items():
    vocab_unicode[key] = "".join((d[b] for b in value))

with open(output_dir/"tiny_train_vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_unicode, f)

with open(output_dir/"tiny_train_merge.txt", "w", encoding="utf-8") as f:
    for left, right in merges:
        left_unicode = "".join((d[b] for b in left))
        right_unicode = "".join((d[b] for b in right))
        f.write(f"{left_unicode}, {right_unicode}\n")


In [20]:
longest_id, longest_token = max(
    vocab.items(),
    key=lambda item: len(item[1]),
)

longest_length = len(longest_token)

longest_display = "".join(
    d[b] for b in longest_token
)

print("ID:", longest_id)
print("Byte length:", longest_length)
print("Serialized token:", longest_display)

ID: 7160
Byte length: 15
Serialized token: Ġaccomplishment


In [21]:
cProfile.run("train_bpe(input_path, vocab_size, special_tokens)", sort="cumtime")

         375545226 function calls (375545118 primitive calls) in 316.072 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       70    0.001    0.000  464.409    6.634 selectors.py:310(select)
       70   12.038    0.172  464.408    6.634 selectors.py:304(_select)
       70  168.627    2.409  334.329    4.776 {built-in method select.select}
        6    0.000    0.000  320.180   53.363 pool.py:500(_wait_for_updates)
        1    0.000    0.000  157.706  157.706 pool.py:738(__exit__)
        1    0.000    0.000  157.706  157.706 pool.py:654(terminate)
        5    0.000    0.000  157.684   31.537 util.py:276(__call__)
        1    0.000    0.000  157.683  157.683 pool.py:680(_terminate_pool)
        1    0.000    0.000  157.677  157.677 pool.py:671(_help_stuff_finish)
        1    0.000    0.000  157.677  157.677 {method 'acquire' of '_multiprocessing.SemLock' objects}
        8    0.000    0.000  157.677   19.710 connectio